# Flujo de Poiseuille con Physics-Informed Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/navier_stokes_pinns/notebooks/01_poiseuille.ipynb)

## 📚 Introducción

El **flujo de Poiseuille** es el flujo laminar entre dos placas paralelas. Es uno de los pocos casos de Navier-Stokes con **solución analítica conocida**, lo que lo hace perfecto para validar PINNs.

### Ecuación Diferencial

Para flujo estacionario en 2D entre placas:

$$
\mu \frac{\partial^2 u}{\partial y^2} = \frac{\partial p}{\partial x}
$$

### Condiciones de Frontera

- $u(y = -H) = 0$ (pared inferior, no-slip)
- $u(y = +H) = 0$ (pared superior, no-slip)

### Solución Analítica

$$
u(y) = -\frac{1}{2\mu} \frac{dp}{dx} (H^2 - y^2)
$$

Perfil parabólico con velocidad máxima en el centro.

## 🔧 Setup

Si estás en **Google Colab**, ejecuta la siguiente celda para instalar dependencias:

In [ ]:
# Descomenta si estás en Google Colab
# !pip install deepxde tensorflow numpy matplotlib seaborn -q

In [ ]:
import sys
import os

# Añadir path del proyecto (si estás local)
if os.path.exists('../src'):
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de plotting
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Librerías cargadas")

## 🧮 Parámetros del Problema

In [ ]:
# Parámetros físicos
H = 0.5          # Medio ancho del canal [m]
L = 2.0          # Longitud del canal [m]
dp_dx = -1.0     # Gradiente de presión [Pa/m] (negativo → flujo positivo)
mu = 0.01        # Viscosidad dinámica [Pa·s]
rho = 1.0        # Densidad [kg/m³]

# Derivados
nu = mu / rho    # Viscosidad cinemática
u_max = -dp_dx * H**2 / (2 * mu)  # Velocidad máxima (en y=0)
Re = rho * u_max * H / mu         # Número de Reynolds

print(f"Parámetros de simulación:")
print(f"  H = {H} m")
print(f"  L = {L} m")
print(f"  dp/dx = {dp_dx} Pa/m")
print(f"  μ = {mu} Pa·s")
print(f"  ν = {nu:.6f} m²/s")
print(f"  u_max = {u_max:.4f} m/s")
print(f"  Re = {Re:.2f}")

## 🏗️ Construir el PINN

Usamos DeepXDE para definir el problema:

In [ ]:
import deepxde as dde

# Definir la ecuación diferencial
def pde(x, u):
    """
    μ * ∂²u/∂y² = dp/dx
    """
    du_dyy = dde.grad.hessian(u, x, i=1, j=1)  # Segunda derivada en y
    return mu * du_dyy - dp_dx

# Geometría
geom = dde.geometry.Rectangle([0, -H], [L, H])

# Condiciones de frontera (no-slip)
def boundary_bottom(x, on_boundary):
    return on_boundary and np.isclose(x[1], -H)

def boundary_top(x, on_boundary):
    return on_boundary and np.isclose(x[1], H)

bc_bottom = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_bottom)
bc_top = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_top)

# Problema completo
data = dde.data.PDE(
    geom,
    pde,
    [bc_bottom, bc_top],
    num_domain=2000,    # Puntos de colocación en el dominio
    num_boundary=200,   # Puntos en fronteras
    num_test=500
)

# Red neuronal: [x, y] → u
net = dde.nn.FNN(
    [2, 50, 50, 50, 1],  # 2 inputs, 3 capas ocultas de 50 neuronas, 1 output
    "tanh",               # Activación
    "Glorot uniform"      # Inicialización
)

# Modelo
model = dde.Model(data, net)

print("✓ Modelo construido")

## 🎯 Entrenar el PINN

Entrenamiento en dos fases:
1. **Adam**: Optimizador adaptativo rápido
2. **L-BFGS**: Optimizador quasi-Newton para refinamiento

In [ ]:
# Fase 1: Adam
model.compile("adam", lr=1e-3, metrics=["l2 relative error"])
losshistory, train_state = model.train(iterations=10000, display_every=1000)

# Fase 2: L-BFGS
model.compile("L-BFGS")
losshistory, train_state = model.train()

print("\n✓ Entrenamiento completado")

## 📊 Visualizar Convergencia

In [ ]:
# Historia de loss
dde.saveplot(losshistory, train_state, issave=False, isplot=True)
plt.show()

## 🔍 Comparar con Solución Analítica

In [ ]:
# Solución analítica
def analytical_solution(y):
    return -dp_dx / (2 * mu) * (H**2 - y**2)

# Evaluar en línea vertical en x = L/2
y_line = np.linspace(-H, H, 100)
x_line = np.full_like(y_line, L/2)
points = np.column_stack([x_line, y_line])

u_pred = model.predict(points)
u_exact = analytical_solution(y_line)

# Plot comparación
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Perfil de velocidad
ax1.plot(u_exact, y_line, 'b-', linewidth=2.5, label='Analítico', alpha=0.7)
ax1.plot(u_pred, y_line, 'r--', linewidth=2, label='PINN', alpha=0.8)
ax1.set_xlabel('Velocidad u [m/s]', fontsize=12)
ax1.set_ylabel('Posición y [m]', fontsize=12)
ax1.set_title('Perfil de Velocidad', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Error puntual
error = np.abs(u_pred.flatten() - u_exact)
ax2.plot(error, y_line, 'g-', linewidth=2)
ax2.set_xlabel('Error Absoluto', fontsize=12)
ax2.set_ylabel('Posición y [m]', fontsize=12)
ax2.set_title('Error del PINN', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Calcular error L2 relativo
l2_error = np.linalg.norm(u_pred.flatten() - u_exact) / np.linalg.norm(u_exact) * 100
ax2.text(0.6, 0.95, f'Error L2: {l2_error:.3f}%',
         transform=ax2.transAxes, fontsize=11,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"\nError L2 relativo: {l2_error:.4f}%")
print(f"Error máximo: {error.max():.6f} m/s")
print(f"Error promedio: {error.mean():.6f} m/s")

## 🌊 Campo de Velocidad Completo

In [ ]:
# Crear malla para visualización
x = np.linspace(0, L, 100)
y = np.linspace(-H, H, 100)
X, Y = np.meshgrid(x, y)

# Predecir velocidad
points_full = np.column_stack([X.flatten(), Y.flatten()])
u_full = model.predict(points_full).reshape(X.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))

# Contour de velocidad
contour = ax.contourf(X, Y, u_full, levels=30, cmap='viridis')
plt.colorbar(contour, ax=ax, label='Velocidad u [m/s]')

# Líneas de contorno
ax.contour(X, Y, u_full, levels=10, colors='white', linewidths=0.5, alpha=0.4)

ax.set_xlabel('x [m]', fontsize=12)
ax.set_ylabel('y [m]', fontsize=12)
ax.set_title('Campo de Velocidad - Flujo de Poiseuille', fontsize=14, fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 🎓 Conclusiones

**Lo que aprendimos:**

1. ✅ El PINN reproduce la solución analítica con alta precisión (<< 1% error)
2. ✅ No necesitamos mallar el dominio manualmente
3. ✅ La red aprende física (ecuación diferencial) + datos (condiciones de frontera)

**Limitaciones:**

- Para casos más complejos (turbulencia, 3D, no estacionarios) el PINN puede requerir más puntos y entrenamiento
- CFD tradicional sigue siendo más robusto para validación industrial

**Siguiente paso:** Lid-Driven Cavity Flow (notebook 02) sin solución analítica 🚀